In [0]:
%python
spark.catalog.setCurrentCatalog("first_project")

In [0]:
%python
df_orders = spark.read.table("silver.orders")
df_orderrows = spark.read.table("silver.orderrows")

df_orders.createOrReplaceTempView("temp_orders")
df_orderrows.createOrReplaceTempView("temp_orderrows")

In [0]:
%python

spark.sql(
    """
    with first_buy as (
      select a.customerkey,
            min(orderdate) as first_buy_date
      from temp_orders a
      group by a.customerkey
    )

    select a.orderdate,
           count(distinct a.orderkey) as total_order_by_date,
           count(distinct a.customerkey) as total_distinct_customer_by_date,
           count(distinct b.productkey) as total_distinct_product_by_date,
           count(c.customerkey) as total_new_customers_by_date,
           round(sum(b.total_price_usd), 2) as total_revenue_usd_by_date,
           sum(b.quantity) as total_quantity_by_date
    from temp_orders as a
    join temp_orderrows b on a.orderkey = b.orderkey
    left join first_buy c on a.customerkey = c.customerkey and a.orderdate = c.first_buy_date
    where 1=1
    and a.orderdate = '2020-10-28'
    --and a.orderdate >= add_months(trunc(current_date(), 'year'), -24)
    group by a.orderdate
    """
).display()

In [0]:
"""with first_buy as (
  select a.customerkey,
        min(orderdate) as first_buy_date
  from first_project.silver.orders a
  group by a.customerkey
),

total_orders_by_date as (
  select a.orderdate,
        count(*) as total_order_by_date,
        count(distinct a.customerkey) as total_distinct_customer_by_date,
        count(distinct b.ProductKey) as total_distinct_product_by_date,
        count(c.customerkey) as total_new_customers_by_date,
        round(sum(b.total_price_usd), 2) as total_revenue_usd_by_date,
        sum(b.quantity) as total_quantity_by_date
  from first_project.silver.orders a
  join first_project.silver.orderrows b on a.orderkey = b.OrderKey
  left join first_buy c on a.customerkey = c.customerkey and a.orderdate = c.first_buy_date
  where a.orderdate > add_months(trunc(current_date(),'year'),-24)
  group by a.orderdate
)

select a.*
from total_orders_by_date a
"""

In [0]:
%python

from pyspark.sql import functions as f

df_orderrows = spark.read.table("silver.orderrows")
df_orders = spark.read.table("silver.orders")
df_first_buy = (
  df_orders
    .groupBy('customerkey')
    .agg(f.min('orderdate').alias('first_buy_date'))
)

(
  df_orders
    .alias('a')
    .join(
      df_orderrows.alias('b'), 
      on=f.col('a.orderkey') == f.col('b.orderkey'), 
      how='inner'
    )
    .join(
      df_first_buy.alias('c'),
      on=[
        f.col('a.customerkey') == f.col('c.customerkey'),
        f.col('a.orderdate') == f.col('c.first_buy_date')
      ], 
      how='left'
    )
    .where(f.col('orderdate') == '2020-10-28')
    .groupBy('orderdate')
    .agg(
      f.countDistinct(f.col('a.orderkey')).alias('total_order_by_date'),
      f.countDistinct(f.col('b.productkey')).alias('total_distinct_product_by_date'),
      f.count(f.col('c.customerkey')).alias('total_new_customers_by_date'),
      f.round(f.sum(f.col('b.total_price_usd')), 2).alias('total_revenue_usd_by_date'),
      f.round(f.sum(f.col('b.quantity')), 2).alias('total_quantity_by_date')
    )
    .display()
)